In [ ]:
import pandas as pd

Specify the file path

In [ ]:
file_path = "https://raw.github.com/jan-source/ppa_erp_student/main/sessions/2_session/"

#### Use ";" as the separator:
data_a = pd.read_csv('churn_part_a.csv', delimiter = ';')

In [ ]:
data_a = pd.read_csv(f'{file_path}churn_part_a.csv')

In [ ]:
data_a.head()

In [ ]:
data_b = pd.read_csv(f'{file_path}churn_part_b.csv')

In [ ]:
data_b.head()

### Combine datasets
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html

In [ ]:
data = pd.merge(data_a, data_b, how='inner', on='CustomerId')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data_a.shape

In [ ]:
data_b.shape

### %-Anteil der Kunden die uns verlassen haben

In [ ]:
data["Exited"].sum()/data.shape[0]

### Determine the number of complete records

In [ ]:
data.dropna(axis=0).shape

### Remove rows where either Gender or Age, or both, are NaN

In [ ]:
data.dropna(axis=0, subset=['Gender', 'Age']).shape

In [ ]:
data.shape

In [ ]:
### Create a dataset with rows containing NaNs removed

In [ ]:
data_cleaned = data.dropna(axis=0)

In [ ]:
data_cleaned.shape

### Visual analysis

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(data_cleaned['CreditScore'], data_cleaned['Exited'])

In [ ]:
data_cleaned.groupby('Exited')['CreditScore'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].min()

### 1. Prediction model: logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

### Selektiere Features (Variablen die für die Vorhersage genutzt werden)

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance']]

In [ ]:
random_variable = 1234

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

### Zeige Koeffizienten formatiert an

In [ ]:
import numpy as np

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Calculate the probability that a customer churns (Exited = 1)

In [ ]:
log_reg_simple.predict_proba(x)

Each customer receives a probability that they stay with us (Exited=0) and a probability that they churn (Exited=1)

### Predict whether a customer churns (1) or stays (0); each number represents a customer


In [ ]:
log_reg_simple.predict(x)

### Compute confusion matrix

In [ ]:
predictions = log_reg_simple.predict(x)

confusion_matrix(observed values, predicted values, ordering)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_cleaned['Exited'], 
                    predictions, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
accuracy = (113 + 7779)/(113 + 1923 + 181 + 7779)

In [ ]:
accuracy

In [ ]:
precision = 113/(113 + 181)

In [ ]:
precision

### Model with more variables

In [ ]:
data_cleaned.head()

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance', 'EstimatedSalary']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

In [ ]:
predictions = log_reg_simple.predict(x)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_cleaned['Exited'], 
                    predictions, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
112/(112+ 176)

### Accuracy calculation using the implemented solution

In [ ]:
metrics.accuracy_score(data_cleaned['Exited'], 
                    predictions)

### Precision calculation using the implemented solution

In [ ]:
metrics.precision_score(data_cleaned['Exited'], 
                    predictions)

### Create a column that takes the value True for women and False for men

In [ ]:
data_cleaned['Gender_Boolean'] = data['Gender'] == 'Female'

In [ ]:
data_cleaned.head()

### Convert to numeric values

In [ ]:
data_cleaned['Gender_numeric'] = data_cleaned['Gender_Boolean'].astype(int)

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned['Geography'].unique()

### Generate dummy encoding for the Geography column

In [ ]:
geography_dummies = pd.get_dummies(data_cleaned['Geography'], prefix="g")

In [ ]:
geography_dummies

### Merge data_cleaned and geography_dummies (add columns to data_cleaned)

In [ ]:
data_full = pd.concat([data_cleaned, geography_dummies], axis=1)

In [ ]:
data_full

### New model with country of origin and gender

In [ ]:
features = ['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']

In [ ]:
x_full = data_full[['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x_full, data_full['Exited'])

In [ ]:
predictions_full = log_reg_simple.predict(x_full)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_full['Exited'], 
                    predictions_full, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_full)

In [ ]:
metrics.precision_score(data_full['Exited'], 
                    predictions_full)

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Decision Tree

In [ ]:
from sklearn import tree

Decision tree with 3 levels (depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=3)

Decision tree with 20 levels (depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=20)

### Fit the tree to the data (training)

In [ ]:
clf_tree.fit(x_full, data_full['Exited'])

### Generate predictions

In [ ]:
predictions_tree = clf_tree.predict(x_full)

In [ ]:
cnf_matrix_tree = metrics.confusion_matrix(
                    data_full['Exited'], 
                    predictions_tree, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix_tree

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_tree)

In [ ]:
metrics.precision_score(data_full['Exited'], 
                    predictions_tree)

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_tree)

- Very high accuracy, but the tree fits the training data too closely (overfitting)
- The tree will not perform as well on new data

### Random forest
Create multiple trees and use the average probability as the prediction

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### Random forest for which we set that each leaf/group must contain at least 20 observations (parameter: min_samples_leaf)

In [ ]:
rf = RandomForestClassifier(random_state = random_variable, min_samples_leaf=20)

In [ ]:
rf.fit(x_full, data_full['Exited'])

In [ ]:
predictions_rf = rf.predict(x_full)

In [ ]:
cnf_rf = metrics.confusion_matrix(data_full['Exited'], predictions_rf, labels=[True, False])

In [ ]:
cnf_rf

### Feature importance

In [ ]:
importances = rf.feature_importances_

In [ ]:
importances

In [ ]:
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

In [ ]:
forest_importances = pd.Series(importances, index=features)

fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

## Gradient boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
gbm = GradientBoostingClassifier(random_state = random_variable, n_estimators=100)

In [ ]:
gbm.fit(x_full, data_full['Exited'])

In [ ]:
predictions_gbm = gbm.predict(x_full)

In [ ]:
cnf_gbm = metrics.confusion_matrix(data_full['Exited'], predictions_gbm, labels=[True, False])

## Shapley Values

In [ ]:
import shap
from random import randrange


In [ ]:
explainer = shap.TreeExplainer(gbm, model_output='probability', data=x_full[features].astype(int).values)

In [ ]:
shap_values = explainer.shap_values(x_full[features])

In [ ]:
shap.initjs()

In [ ]:
shap.summary_plot(shap_values, x_full[features])

In [ ]:
explainer.expected_value

“How much did this feature contribute to changing the model’s prediction from the average prediction?”

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[2], x_full.iloc[2,:][features])

In [ ]:
shap_object = shap.Explanation(
        base_values =   explainer.expected_value,
        values = shap_values[2],
        feature_names = features,
        data = shap_values[0].data)

shap.plots.waterfall(shap_object)